## Create Key Pair
Creates an EC2 key pair on AWS and saves the private key locally as a `.pem` file for SSH access.

In [3]:
import boto3

ec2 = boto3.client('ec2', region_name='ap-south-2')

key_pair_name = 'demo-key-value'

response = ec2.create_key_pair(KeyName=key_pair_name)

private_key = response['KeyMaterial']

# Save the private key to a .pem file (keep this secure!)
pem_file = f'{key_pair_name}.pem'
with open(pem_file, 'w') as f:
    f.write(private_key)

print(f"Key pair '{key_pair_name}' created successfully.")
print(f"Private key saved to: {pem_file}")
print(f"Key Pair ID: {response['KeyPairId']}")


Key pair 'demo-key-value' created successfully.
Private key saved to: demo-key-value.pem
Key Pair ID: key-0aaac580d242e98a1


## Delete Key Pair
Deletes the EC2 key pair from AWS and removes the local `.pem` file.

In [2]:
import os

# Delete the key pair from AWS
ec2.delete_key_pair(KeyName=key_pair_name)
print(f"Key pair '{key_pair_name}' deleted from AWS.")

# Remove the local .pem file
if os.path.exists(pem_file):
    os.remove(pem_file)
    print(f"Local file '{pem_file}' deleted.")
else:
    print(f"Local file '{pem_file}' not found.")

Key pair 'demo-key-value' deleted from AWS.
Local file 'demo-key-value.pem' deleted.


## 1. Create an EC2 Instance
Defines configuration and creates an EC2 instance using `run_instances`.

In [7]:
import boto3

ec2 = boto3.client('ec2', region_name='ap-south-2')

response = ec2.run_instances(
    ImageId='ami-011bcca8b6d63b21c',  # AMI ID (OS image) — replace with a valid AMI for your region
    InstanceType='t3.micro',          # Nitro-based instance (required for UEFI AMIs; also free-tier eligible)
    MinCount=1,                       # Minimum number of instances to launch
    MaxCount=1,                       # Maximum number of instances to launch
    KeyName='demo-key-value',         # Key pair name used for SSH access
    TagSpecifications=[{
        'ResourceType': 'instance',
        'Tags': [{'Key': 'Name', 'Value': 'MyEC2Instance'}]  # Tag to identify the instance
    }]
)

instance_id = response['Instances'][0]['InstanceId']
print(f"Instance created: {instance_id}")

Instance created: i-00952ce885aa2f757


## 2. Launch an EC2 Instance (Wait Until Running)
Waits for the instance to reach the `running` state after creation.

In [8]:
ec2_resource = boto3.resource('ec2', region_name='ap-south-2')

# Get the instance object using the instance ID from the previous cell
instance = ec2_resource.Instance(instance_id)

# Block until the instance transitions to 'running' state
instance.wait_until_running()

# Reload attributes to get the latest public IP and DNS after startup
instance.reload()

print(f"Instance is running.")
print(f"Public IP  : {instance.public_ip_address}")  # Use this IP to SSH into the instance
print(f"Public DNS : {instance.public_dns_name}")     # DNS hostname for the instance

Instance is running.
Public IP  : 16.112.58.68
Public DNS : ec2-16-112-58-68.ap-south-2.compute.amazonaws.com


## 3. Create a Security Group
A security group acts as a virtual firewall — controls allowed inbound and outbound traffic.

In [9]:
# Create a new security group in the default VPC
sg_response = ec2.create_security_group(
    GroupName='demo-sg',                         # Unique name for the security group
    Description='Security group for demo EC2'    # Human-readable description
)

sg_id = sg_response['GroupId']  # Unique ID assigned to the new security group
print(f"Security Group created: {sg_id}")

# Add an inbound rule to allow SSH (port 22) from any IP address
ec2.authorize_security_group_ingress(
    GroupId=sg_id,
    IpPermissions=[{
        'IpProtocol': 'tcp',                         # Protocol (tcp/udp/icmp)
        'FromPort': 22,                              # Start of port range
        'ToPort': 22,                                # End of port range
        'IpRanges': [{'CidrIp': '0.0.0.0/0'}]       # CIDR — restrict to your IP in production
    }]
)
print("Inbound SSH rule (port 22) added.")

Security Group created: sg-060bacb03c5737090
Inbound SSH rule (port 22) added.


## 4. Attach a Security Group to an EC2 Instance
Adds an existing security group to a running or stopped instance.

In [10]:
# Reload the instance to get its current security groups
instance = ec2_resource.Instance(instance_id)
current_sg_ids = [sg['GroupId'] for sg in instance.security_groups]  # Existing SG IDs

# Append the new security group to the existing list
current_sg_ids.append(sg_id)

# modify_instance_attribute replaces all SGs — so pass the full updated list
ec2.modify_instance_attribute(
    InstanceId=instance_id,   # Target EC2 instance
    Groups=current_sg_ids     # Full list including the newly attached SG
)
print(f"Security group '{sg_id}' attached to instance '{instance_id}'.")

Security group 'sg-060bacb03c5737090' attached to instance 'i-00952ce885aa2f757'.


## 5. Detach a Security Group from an EC2 Instance
Removes a specific security group from an instance (at least one must remain attached).

In [11]:
# Reload to get the latest list of attached security groups
instance.reload()
current_sg_ids = [sg['GroupId'] for sg in instance.security_groups]

# Filter out the security group to be detached
updated_sg_ids = [gid for gid in current_sg_ids if gid != sg_id]

if not updated_sg_ids:
    # EC2 requires at least one security group to remain attached at all times
    print("Cannot detach: an instance must have at least one security group.")
else:
    ec2.modify_instance_attribute(
        InstanceId=instance_id,   # Target EC2 instance
        Groups=updated_sg_ids     # Updated list with the target SG removed
    )
    print(f"Security group '{sg_id}' detached from instance '{instance_id}'.")

Security group 'sg-060bacb03c5737090' detached from instance 'i-00952ce885aa2f757'.


## 6. Check EC2 Instance Status
Retrieves the current state and health checks (system + instance level) for an EC2 instance.

In [12]:
status_response = ec2.describe_instance_status(
    InstanceIds=[instance_id],    # List of instance IDs to query
    IncludeAllInstances=True      # Include instances not in 'running' state (e.g. stopped)
)

for status in status_response['InstanceStatuses']:
    print(f"Instance ID      : {status['InstanceId']}")
    print(f"Instance State   : {status['InstanceState']['Name']}")      # running / stopped / terminated
    print(f"System Status    : {status['SystemStatus']['Status']}")     # AWS hardware health check
    print(f"Instance Status  : {status['InstanceStatus']['Status']}")  # OS-level health check

Instance ID      : i-00952ce885aa2f757
Instance State   : running
System Status    : ok
Instance Status  : ok


## 7. Start, Stop, and Terminate an EC2 Instance
Controls the lifecycle of an EC2 instance — start resumes billing, stop pauses it, terminate permanently deletes it.

In [15]:
# --- START a stopped instance ---
# Resumes the instance; billing restarts for compute time
# ec2.start_instances(InstanceIds=[instance_id])
# print(f"Instance '{instance_id}' is starting...")

# --- STOP a running instance ---
# Pauses the instance; compute billing stops, but EBS storage cost continues
# ec2.stop_instances(InstanceIds=[instance_id])
# print(f"Instance '{instance_id}' is stopping...")

# --- TERMINATE an instance (IRREVERSIBLE) ---
# Permanently deletes the instance and its ephemeral storage
ec2.terminate_instances(InstanceIds=[instance_id])
print(f"Instance '{instance_id}' is terminating...")

Instance 'i-00952ce885aa2f757' is terminating...
